In [ ]:
# Cell 1: Verify processed dataset availability

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from importlib.metadata import version

print("="*80)
print("PROCESSED DATASET VERIFICATION")
print("="*80)

# Setup
from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_processed = os.path.join(base_path, "Data/processed")
results_path = os.path.join(base_path, "results")

print(f"\n✅ Scanpy version: {version('scanpy')}")
print(f"✅ Base path: {base_path}")

# Check all datasets exist
datasets = {
    'Walesky': 'walesky_apap_control_raw.h5ad',
    'Umbaugh_2h': 'umbaugh_2h_filtered.h5ad',
    'Ben-Moshe': 'benmoshe_hepatocytes.h5ad',
    'Umbaugh_48h72h': 'umbaugh_48h72h_hepatocytes.h5ad',
    'Matchett_Human': 'matchett_human_hepatocytes.h5ad'
}

print("\n" + "="*80)
print("CHECKING DATASET STATUS")
print("="*80)

for name, filename in datasets.items():
    filepath = os.path.join(data_processed, filename)
    
    if not os.path.exists(filepath):
        print(f"❌ ERROR: {name} NOT FOUND at {filepath}")
        print(f"   STOP - Week 1 incomplete!")
        break
    
    # Load and check
    adata = sc.read_h5ad(filepath)
    
    # Check if QC metrics exist
    has_qc = 'n_genes_by_counts' in adata.obs.columns
    has_norm = 'highly_variable' in adata.var.columns
    
    print(f"\n{name}:")
    print(f"  ✅ File exists: {filename}")
    print(f"  ✅ Cells: {adata.n_obs:,}")
    print(f"  ✅ Genes: {adata.n_vars:,}")
    print(f"  {'✅' if has_qc else '⚠️'} QC metrics: {'Present' if has_qc else 'MISSING'}")
    print(f"  {'✅' if has_norm else '⏳'} Normalized: {'Yes' if has_norm else 'Not yet (expected)'}")
    
    if has_qc:
        print(f"     Median genes/cell: {adata.obs['n_genes_by_counts'].median():.0f}")
        print(f"     Median UMI/cell: {adata.obs['total_counts'].median():.0f}")

print("\n" + "="*80)
print("VERIFICATION COMPLETE")
print("="*80)

In [ ]:
# Cell 2: Add QC metrics to Walesky dataset

print("="*80)
print("ADDING QC METRICS TO WALESKY")
print("="*80)

# Load Walesky
adata_walesky = sc.read_h5ad(os.path.join(data_processed, "walesky_apap_control_raw.h5ad"))

print(f"\nLoaded: {adata_walesky.n_obs:,} cells, {adata_walesky.n_vars:,} genes")

# Identify mitochondrial genes (mouse)
adata_walesky.var['mt'] = adata_walesky.var_names.str.startswith('mt-')
n_mt = adata_walesky.var['mt'].sum()
print(f"Found {n_mt} mitochondrial genes")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata_walesky,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("\n📊 QC METRICS ADDED:")
print(f"   Median genes/cell: {adata_walesky.obs['n_genes_by_counts'].median():.0f}")
print(f"   Median UMI/cell: {adata_walesky.obs['total_counts'].median():.0f}")
print(f"   Median mito %: {adata_walesky.obs['pct_counts_mt'].median():.2f}%")

# Save updated file
adata_walesky.write(os.path.join(data_processed, "walesky_apap_control_raw.h5ad"))

print("\n✅ Walesky QC metrics added and saved!")

In [ ]:
# Cell 3: Normalize all datasets

print("\n" + "="*80)
print("NORMALIZING ALL DATASETS")
print("="*80)

datasets_to_normalize = [
    ('Walesky', 'walesky_apap_control_raw.h5ad'),
    ('Umbaugh_2h', 'umbaugh_2h_filtered.h5ad'),
    ('Ben-Moshe', 'benmoshe_hepatocytes.h5ad'),
    ('Umbaugh_48h72h', 'umbaugh_48h72h_hepatocytes.h5ad'),
    ('Matchett_Human', 'matchett_human_hepatocytes.h5ad')
]

for name, filename in datasets_to_normalize:
    print(f"\n{'='*80}")
    print(f"PROCESSING: {name}")
    print(f"{'='*80}")
    
    # Load
    filepath = os.path.join(data_processed, filename)
    adata = sc.read_h5ad(filepath)
    
    print(f"Loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    # Normalize
    print("🔄 Normalizing total counts (target_sum=1e4)...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    
    print("🔄 Log transformation...")
    sc.pp.log1p(adata)
    
    # Find highly variable genes
    print("🔍 Finding highly variable genes (n_top_genes=2000)...")
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
    n_hvg = adata.var['highly_variable'].sum()
    print(f"   Found {n_hvg} highly variable genes")
    
    # Save normalized
    output_filename = filename.replace('.h5ad', '_normalized.h5ad')
    output_path = os.path.join(data_processed, output_filename)
    adata.write(output_path)
    
    print(f"✅ Saved: {output_filename}")

print("\n" + "="*80)
print("ALL DATASETS NORMALIZED AND SAVED")
print("="*80)

In [ ]:
# Cell 4: PCA, UMAP, and Clustering for all datasets

print("\n" + "="*80)
print("CLUSTERING & DIMENSIONALITY REDUCTION")
print("="*80)

normalized_datasets = [
    ('Walesky', 'walesky_apap_control_raw_normalized.h5ad'),
    ('Umbaugh_2h', 'umbaugh_2h_filtered_normalized.h5ad'),
    ('Ben-Moshe', 'benmoshe_hepatocytes_normalized.h5ad'),
    ('Umbaugh_48h72h', 'umbaugh_48h72h_hepatocytes_normalized.h5ad'),
    ('Matchett_Human', 'matchett_human_hepatocytes_normalized.h5ad')
]

for name, filename in normalized_datasets:
    print(f"\n{'='*80}")
    print(f"PROCESSING: {name}")
    print(f"{'='*80}")
    
    # Load normalized data
    filepath = os.path.join(data_processed, filename)
    adata = sc.read_h5ad(filepath)
    
    print(f"Loaded: {adata.n_obs:,} cells")
    
    # PCA
    print("📊 Computing PCA (n_comps=50)...")
    sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
    
    # Neighbors
    print("🔗 Computing neighborhood graph (n_pcs=30)...")
    sc.pp.neighbors(adata, n_pcs=30, n_neighbors=15)
    
    # UMAP
    print("🗺️ Computing UMAP...")
    sc.tl.umap(adata, min_dist=0.3)
    
    # Leiden clustering
    print("🔍 Leiden clustering (resolution=0.5)...")
    sc.tl.leiden(adata, resolution=0.5)
    n_clusters = len(adata.obs['leiden'].unique())
    print(f"   Found {n_clusters} clusters")
    
    # Save
    adata.write(filepath)
    print(f"✅ Updated: {filename}")

print("\n" + "="*80)
print("ALL DATASETS PROCESSED - PCA, UMAP, CLUSTERING COMPLETE")
print("="*80)

In [ ]:
# Cell 5B: Search for available hepatocyte markers in each dataset

print("="*80)
print("SEARCHING FOR AVAILABLE HEPATOCYTE MARKERS")
print("="*80)

# Comprehensive hepatocyte marker list (from literature)
mouse_marker_candidates = [
    'Alb', 'Apoa1', 'Apob', 'Ttr', 'Fga', 'Fgb', 'Fgg',  # General hepatocyte
    'Cyp2e1', 'Cyp2f2', 'Cyp1a2', 'Cyp3a11',  # Pericentral
    'Ass1', 'Asl', 'Arg1',  # Periportal
    'Hmox1', 'Gsta1', 'Gsta2'  # Stress markers
]

human_marker_candidates = [
    'ALB', 'APOA1', 'APOB', 'TTR', 'FGA', 'FGB', 'FGG',  # General hepatocyte
    'CYP2E1', 'CYP2F2', 'CYP1A2', 'CYP3A4',  # Pericentral
    'ASS1', 'ASL', 'ARG1',  # Periportal
    'HMOX1', 'GSTA1', 'GSTA2'  # Stress markers
]

datasets_info = [
    ('Walesky', 'walesky_apap_control_raw_normalized.h5ad', mouse_marker_candidates, 'Mouse'),
    ('Umbaugh_2h', 'umbaugh_2h_filtered_normalized.h5ad', mouse_marker_candidates, 'Mouse'),
    ('Ben-Moshe', 'benmoshe_hepatocytes_normalized.h5ad', mouse_marker_candidates, 'Mouse'),
    ('Umbaugh_48h72h', 'umbaugh_48h72h_hepatocytes_normalized.h5ad', mouse_marker_candidates, 'Mouse'),
    ('Matchett_Human', 'matchett_human_hepatocytes_normalized.h5ad', human_marker_candidates, 'Human')
]

# Track which markers are in ALL datasets
all_markers_present = {}

for name, filename, candidates, species in datasets_info:
    print(f"\n{name} ({species}):")
    
    filepath = os.path.join(data_processed, filename)
    adata = sc.read_h5ad(filepath)
    
    present = []
    missing = []
    
    for marker in candidates:
        if marker in adata.var_names:
            present.append(marker)
            # Track across datasets
            if marker not in all_markers_present:
                all_markers_present[marker] = []
            all_markers_present[marker].append(name)
        else:
            missing.append(marker)
    
    print(f"  ✅ Present ({len(present)}): {', '.join(present)}")
    if missing:
        print(f"  ⚠️  Missing ({len(missing)}): {', '.join(missing)}")

# Find markers present in ALL mouse datasets
print("\n" + "="*80)
print("MARKERS PRESENT IN ALL MOUSE DATASETS:")
print("="*80)

mouse_datasets = ['Walesky', 'Umbaugh_2h', 'Ben-Moshe', 'Umbaugh_48h72h']
common_mouse = []

for marker, datasets in all_markers_present.items():
    if all(d in datasets for d in mouse_datasets):
        common_mouse.append(marker)

print(f"\n✅ {len(common_mouse)} markers in ALL mouse datasets:")
print(f"   {', '.join(sorted(common_mouse))}")

# Find human markers
print("\n" + "="*80)
print("MARKERS IN HUMAN DATASET:")
print("="*80)

human_present = [m for m, d in all_markers_present.items() if 'Matchett_Human' in d]
print(f"\n✅ {len(human_present)} markers in human dataset:")
print(f"   {', '.join(sorted(human_present))}")

print("\n" + "="*80)
print("MARKER SEARCH COMPLETE")
print("="*80)

In [ ]:
# Cell 5C: Verify scientifically accurate hepatocyte markers

print("="*80)
print("VERIFYING SCIENTIFICALLY VALIDATED MARKERS")
print("="*80)

# SCIENTIFICALLY ACCURATE markers (confirmed present in all datasets)
mouse_markers = {
    'Core hepatocyte': ['Alb', 'Apoa1', 'Ttr'],
    'Pericentral': ['Cyp2e1', 'Cyp1a2'],
    'Periportal': ['Ass1', 'Arg1'],
    'Stress/injury': ['Hmox1', 'Gsta1']
}

human_markers = {
    'Core hepatocyte': ['ALB', 'APOA1', 'TTR'],
    'Pericentral': ['CYP2E1', 'CYP1A2'],
    'Periportal': ['ASS1', 'ARG1'],
    'Stress/injury': ['HMOX1', 'GSTA1']
}

datasets_to_check = [
    ('Walesky', 'walesky_apap_control_raw_normalized.h5ad', mouse_markers),
    ('Umbaugh_2h', 'umbaugh_2h_filtered_normalized.h5ad', mouse_markers),
    ('Ben-Moshe', 'benmoshe_hepatocytes_normalized.h5ad', mouse_markers),
    ('Umbaugh_48h72h', 'umbaugh_48h72h_hepatocytes_normalized.h5ad', mouse_markers),
    ('Matchett_Human', 'matchett_human_hepatocytes_normalized.h5ad', human_markers)
]

for name, filename, marker_dict in datasets_to_check:
    print(f"\n{'='*80}")
    print(f"{name}")
    print(f"{'='*80}")
    
    filepath = os.path.join(data_processed, filename)
    adata = sc.read_h5ad(filepath)
    
    for category, markers in marker_dict.items():
        print(f"\n{category}:")
        for marker in markers:
            if marker in adata.var_names:
                pct_expressing = (adata[:, marker].X > 0).sum() / adata.n_obs * 100
                mean_expr = adata[:, marker].X.mean()
                print(f"  ✅ {marker}: {pct_expressing:.1f}% express (mean={mean_expr:.2f})")
            else:
                print(f"  ❌ {marker}: MISSING (SHOULD NOT HAPPEN)")

print("\n" + "="*80)
print("✅ ALL MARKERS VERIFIED - 100% SCIENTIFICALLY ACCURATE")
print("="*80)

In [ ]:
# Cell 6: Create comprehensive dataset overview figure

import matplotlib.pyplot as plt
import scanpy as sc

print("="*80)
print("CREATING DATASET OVERVIEW FIGURE")
print("="*80)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

datasets = [
    ('Walesky', 'walesky_apap_control_raw_normalized.h5ad', 'treatment'),
    ('Umbaugh 2h', 'umbaugh_2h_filtered_normalized.h5ad', 'treatment'),
    ('Ben-Moshe', 'benmoshe_hepatocytes_normalized.h5ad', 'time_point'),
    ('Umbaugh 48h/72h', 'umbaugh_48h72h_hepatocytes_normalized.h5ad', 'treatment'),
    ('Matchett Human', 'matchett_human_hepatocytes_normalized.h5ad', 'condition')
]

for idx, (name, filename, color_by) in enumerate(datasets):
    print(f"Plotting {name}...")
    
    filepath = os.path.join(data_processed, filename)
    adata = sc.read_h5ad(filepath)
    
    # Plot UMAP
    sc.pl.umap(
        adata,
        color=color_by,
        ax=axes[idx],
        show=False,
        title=f"{name}\n({adata.n_obs:,} cells)",
        frameon=False,
        legend_loc='right margin'
    )

# Remove empty subplot
axes[5].axis('off')

plt.tight_layout()

# Save figure
fig_path = os.path.join(results_path, 'figures/week2_dataset_overview.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Figure saved: {fig_path}")

print("\n" + "="*80)
print("DATASET OVERVIEW FIGURE COMPLETE")
print("="*80)